# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d06_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # noise train
    shutil.copy(glob.glob("/kaggle/input/**/d06_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # noise test
    print("Kaggle: noise data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: noise data copied to ./drive/MyDrive/Regression/noise_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795363049.434944      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795363067.021956      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (noise_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['A', 'B', 'C', 'D', 'O']
TARGET_LABEL = 'Noise level (Leq)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (168, 6)
First 5 rows of training data:
       A    B    C    D   O    Leq
0   894  354  228   30  42  70.06
1  1068  312  234   24  24  69.14
2  1506  894  198  222  18  73.02
3  1188  498  252  354  18  75.71
4  1068  438  210  366  30  70.78

Shape of test data: (60, 6)
First 5 rows of test data:
       A    B    C    D   O    Leq
0  1032  282   90  108   6  70.07
1  1188  498  252  408  18  72.90
2   570   90   60   84  24  68.66
3   450  156  150   72   0  50.24
4   240   66   18   24   0  40.17


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (168, 5)
Shape of y_train: (168,)
Shape of X_test: (60, 5)
Shape of y_test: (60,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-0.34508703 -0.52871218  0.0597039  -1.05667951  1.31930842]
 [-0.08377773 -0.63289541  0.09903824 -1.09452265  0.25541811]
 [ 0.57400083  0.8107865  -0.13696778  0.15430089 -0.09921199]
 [ 0.09643557 -0.17151253  0.21704125  0.98684991 -0.09921199]
 [-0.08377773 -0.32034572 -0.05829911  1.06253618  0.61004822]]

First five rows of normalized X_test:
[[-0.13784172 -0.707312   -0.84498584 -0.56471872 -0.8084722 ]
 [ 0.09643557 -0.17151253  0.21704125  1.32743815 -0.09921199]
 [-0.83166295 -1.1835782  -1.04165752 -0.71609127  0.25541811]
 [-1.01187626 -1.01986169 -0.45164247 -0.79177755 -1.16310231]
 [-1.32724954 -1.24311147 -1.31699788 -1.09452265 -1.16310231]]


# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/noise-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 19.291167396300658,
  'cv_rmse': 4.3921711483389005,
  'best_params': {'n_estimators': 300,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.3,
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 17.22207284999383,
  'test_rmse': np.float64(4.149948535824731),
  'test_corr_coef': np.float64(0.9650589112564479),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 18.114017364046852,
  'cv_rmse': 4.256056550851604,
  'best_params': {'n_estimators': 300,
   'criterion': 'friedman_mse',
   'max_depth': 20,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.3,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.01,
  

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  70.633874  69.913493  70.319304  71.186465  73.991439  71.124647   
1  72.415998  76.086440  75.072292  74.542243  76.168215  75.539615   
2  49.902224  57.151979  54.730897  57.733003  60.391592  63.581342   
3  56.756522  60.662822  59.360679  64.841923  65.441938  62.081095   
4  38.881303  36.325699  38.851656  39.563004  40.244177  38.337725   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  73.019137  71.544408  73.606628  73.522207  72.887698  74.628316   
1  75.437076  75.497460  75.755961  75.882956  76.693596  76.186756   
2  57.871098  55.972532  55.950780  57.886733  50.293354  62.265733   
3  61.532734  61.077510  65.336904  64.275205  65.551640  65.235322   
4  40.733943  40.667806  35.636491  43.343201  38.723134  37.906789   

        0.95  Actual  
0  72.696340   70.07  
1  76.097822   72.90  
2  62.371355   68.66  
3  67.604639   50.24  
4  39.885962   40.17  


In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 73.33%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 60 of 60 rows (100.0%)


{'empirical_quantiles': [np.float64(0.16666666666666666),
  np.float64(0.31666666666666665),
  np.float64(0.35),
  np.float64(0.43333333333333335),
  np.float64(0.4666666666666667),
  np.float64(0.5166666666666667),
  np.float64(0.5833333333333334),
  np.float64(0.65),
  np.float64(0.7),
  np.float64(0.7666666666666667),
  np.float64(0.7833333333333333),
  np.float64(0.8666666666666667),
  np.float64(0.9)],
 'average_interval_lengths': [np.float64(2.5463848044071415),
  np.float64(4.768607045933781),
  np.float64(6.762400005214496)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.4957178202769414),
 'average_crps': np.float64(1.9039485955159237),
 'nll_values': array([ 2.95786985,  3.593597  ,  5.79679904, 11.16973541,  1.92153447,
         3.75555521,  0.83321359,  2.23945225,  3.87234579,  3.90567557,
         2.29454954,  1.33017713,  1.14485849,  2.31410169,  2.75027065,
         0.66429399,  1.69230356,  0.7263375 ,  2.35566509,  2.70879588,
         0.89808189,

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 66.67%
The empirical coverage after calibration is: 90.00%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  59.494251  66.257172  67.861794  69.708893  69.971252  69.785919   
1  62.885342  73.231758  73.776306  73.974586  75.021324  74.659119   
2  54.070042  60.706970  59.590023  61.674683  54.888935  52.008614   
3  54.542194  55.124985  61.701706  59.834469  61.562916  66.202919   
4  31.938635  29.774456  35.087864  32.997044  37.000103  37.915421   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  70.429840  69.736328  71.696609  71.855896  72.072235  72.457458   
1  76.746674  76.815109  77.607468  76.134888  76.922180  76.920197   
2  52.270248  56.604988  56.147980  67.952423  69.860023  71.894012   
3  65.138977  66.925789  65.166702  66.379791  68.758942  71.894012   
4  41.994995  43.047791  41.393223  59.590351  68.758942  71.894012   

        0.95  Actual  
0  77.585487   70.07  
1  79.081444   72.90  
2  77.585487   68.66  
3  77.585487   50.24  
4  77.585487   40.17  


In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 98.33%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 60 of 60 rows (100.0%)


{'empirical_quantiles': [np.float64(0.016666666666666666),
  np.float64(0.15),
  np.float64(0.16666666666666666),
  np.float64(0.25),
  np.float64(0.36666666666666664),
  np.float64(0.4166666666666667),
  np.float64(0.5833333333333334),
  np.float64(0.7333333333333333),
  np.float64(0.85),
  np.float64(0.8833333333333333),
  np.float64(0.95),
  np.float64(0.9666666666666667),
  np.float64(1.0)],
 'average_interval_lengths': [np.float64(8.861650848388672),
  np.float64(15.486923980712891),
  np.float64(25.006215190887453)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.998066864816569),
 'average_crps': np.float64(2.3363833347479503),
 'nll_values': array([2.62372879, 2.7287754 , 3.65028752, 5.13602338, 3.59893827,
        2.85338705, 2.65042885, 2.65153993, 2.710754  , 3.23856799,
        2.46344728, 2.69009054, 2.65316085, 2.57545311, 3.00205809,
        2.4159974 , 2.49465739, 2.63709927, 2.84369322, 3.39956427,
        2.56355671, 2.51131017, 2.44590604, 3.283253

In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 96.67%
The empirical coverage after calibration is: 83.33%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  68.128090  69.848240  72.679825  72.341410  71.281874  71.519067   
1  74.579661  75.211835  77.053697  75.058361  73.669163  74.414098   
2  59.640543  64.805318  60.609740  60.658476  66.029045  66.160792   
3  59.545508  56.750974  63.486345  61.843618  65.179313  62.219386   
4  32.611078  33.534553  34.475696  36.642762  37.431787  40.339206   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  71.628741  71.283225  71.214862  71.211119  72.698885  71.417270   
1  74.079883  75.782390  73.133687  75.553216  76.934185  74.903587   
2  60.820013  56.445651  54.907893  49.129090  60.133903  62.954889   
3  59.573134  62.616808  60.085763  65.744717  65.380028  66.165092   
4  37.526469  40.793309  42.019111  38.441535  41.012050  37.807903   

        0.95  Actual  
0  70.649406   70.07  
1  74.755271   72.90  
2  63.808156   68.66  
3  68.617798   50.24  
4  49.688242   40.17  


In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 71.67%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 60 of 60 rows (100.0%)


{'empirical_quantiles': [np.float64(0.21666666666666667),
  np.float64(0.2833333333333333),
  np.float64(0.38333333333333336),
  np.float64(0.4166666666666667),
  np.float64(0.45),
  np.float64(0.5166666666666667),
  np.float64(0.6166666666666667),
  np.float64(0.6833333333333333),
  np.float64(0.7666666666666667),
  np.float64(0.8166666666666667),
  np.float64(0.85),
  np.float64(0.9),
  np.float64(0.9333333333333333)],
 'average_interval_lengths': [np.float64(3.1007611236403205),
  np.float64(5.768889599658108),
  np.float64(9.087311323454644)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.387246887325931),
 'average_crps': np.float64(1.7827961739568174),
 'nll_values': array([1.62905418, 2.50783484, 3.75751754, 8.08805616, 2.66940787,
        2.73624136, 0.81011507, 3.42412917, 3.94698652, 3.50797214,
        1.38542325, 1.53579265, 1.24399735, 1.45044072, 2.45321379,
        0.7884198 , 1.77022695, 1.12762156, 2.33874658, 2.49580227,
        0.6577729 , 0.78969

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 63.33%
The empirical coverage after calibration is: 93.33%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=4.0721 val_loss=0.0000 scale=1.0000 norm=10.8994


[iter 100] loss=2.9562 val_loss=0.0000 scale=2.0000 norm=5.1677


        0.05        0.1       0.15        0.2        0.3        0.4  \
0  62.925537  64.181415  65.028750  65.702185  66.798767  67.735755   
1  71.931409  72.849837  73.469496  73.961981  74.763915  75.449138   
2  51.338820  52.614612  53.475383  54.159497  55.273467  56.225313   
3  57.972668  59.187701  60.007478  60.659011  61.719929  62.626444   
4  30.127707  32.097849  33.427094  34.483536  36.203785  37.673674   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  68.611536  69.487317  70.424306  71.520888  72.194322  73.041657   
1  76.089600  76.730062  77.415285  78.217219  78.709705  79.329364   
2  57.114981  58.004649  58.956496  60.070466  60.754580  61.615350   
3  63.473742  64.321041  65.227556  66.288474  66.940007  67.759784   
4  39.047544  40.421414  41.891303  43.611551  44.667993  45.997239   

        0.95  Actual  
0  74.297535   70.07  
1  80.247792   72.90  
2  62.891143   68.66  
3  68.974817   50.24  
4  47.967381   40.17  


In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 86.67%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 60 rows (0.0%)


{'empirical_quantiles': [np.float64(0.08333333333333333),
  np.float64(0.08333333333333333),
  np.float64(0.16666666666666666),
  np.float64(0.21666666666666667),
  np.float64(0.26666666666666666),
  np.float64(0.4),
  np.float64(0.55),
  np.float64(0.75),
  np.float64(0.8166666666666667),
  np.float64(0.8666666666666667),
  np.float64(0.8833333333333333),
  np.float64(0.9),
  np.float64(0.95)],
 'average_interval_lengths': [np.float64(5.868647267148422),
  np.float64(8.936293183709841),
  np.float64(11.469608129642834)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.8093271467404493),
 'average_crps': np.float64(2.252322958801552),
 'nll_values': array([2.24829645, 2.64232146, 7.57927337, 9.95502284, 2.63098627,
        2.57267965, 1.97210924, 1.98256078, 3.53711215, 3.71058921,
        1.99904935, 1.95395679, 2.09763985, 2.34261011, 2.46203035,
        1.83668407, 1.82473064, 1.91043138, 2.30282635, 3.59647283,
        1.92092819, 2.01191649, 2.02354582, 4.5212375

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 80.00%
The empirical coverage after calibration is: 80.00%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  49.804266  63.077135  67.710570  70.928312  70.997731  70.886276   
1  49.804266  63.077135  67.710570  75.435195  75.469997  75.742660   
2  43.238587  49.803733  53.952521  55.538656  50.124657  50.901197   
3  49.804266  60.932135  62.828766  60.035605  61.726844  61.374517   
4  37.156333  34.775522  35.856119  35.082485  36.328591  37.917605   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  71.350070  72.371254  72.262557  71.654647  74.475508  74.508712   
1  77.029596  77.643130  77.156288  77.337127  76.627472  76.922816   
2  54.270213  53.390143  53.517691  66.933888  74.475508  74.508712   
3  61.519995  63.238304  67.242770  66.463189  74.475508  74.508712   
4  36.378593  39.028603  41.935014  63.419996  74.475508  74.508712   

        0.95  Actual  
0  78.742158   70.07  
1  79.519746   72.90  
2  78.742158   68.66  
3  78.742158   50.24  
4  78.742158   40.17  


In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 91.67%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 60 of 60 rows (100.0%)


{'empirical_quantiles': [np.float64(0.08333333333333333),
  np.float64(0.15),
  np.float64(0.16666666666666666),
  np.float64(0.35),
  np.float64(0.4166666666666667),
  np.float64(0.5166666666666667),
  np.float64(0.6333333333333333),
  np.float64(0.75),
  np.float64(0.85),
  np.float64(0.8833333333333333),
  np.float64(0.95),
  np.float64(0.9666666666666667),
  np.float64(1.0)],
 'average_interval_lengths': [np.float64(7.887974594804232),
  np.float64(19.76374306314723),
  np.float64(32.311558698729435)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(3.3077625928738645),
 'average_crps': np.float64(2.681804987306631),
 'nll_values': array([3.10387993, 3.20495086, 4.22634803, 4.11733219, 3.52577229,
        3.1493207 , 3.16489471, 3.10005314, 3.16697752, 3.13012405,
        3.11581569, 3.19901171, 3.09984153, 3.11943593, 3.12561088,
        3.0932962 , 3.1113331 , 3.1177542 , 3.09573577, 3.36229684,
        3.12086103, 3.13204958, 3.11185391, 3.0978163 , 3.13551389,
 

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 93.33%
The empirical coverage after calibration is: 80.00%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  66.298236  63.715955  66.922092  68.631262  70.020615  70.056172   
1  69.870195  73.586813  73.432331  72.877020  74.584773  75.246752   
2  59.782870  58.324387  58.014663  59.601930  59.620015  57.700814   
3  53.934435  56.667996  58.110818  57.596653  59.576981  59.294040   
4  36.107740  33.966397  34.857014  34.474472  36.770945  37.759929   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  67.477802  71.075021  70.532679  71.650373  71.046510  71.665381   
1  75.770810  76.950370  76.204675  76.578250  76.434492  77.816379   
2  62.771121  57.031813  64.894674  62.849329  63.639946  65.577895   
3  55.658456  64.400146  65.525372  63.914731  65.536584  68.095029   
4  37.362054  38.093064  40.333489  42.129116  39.128247  42.719082   

        0.95  Actual  
0  72.830461   70.07  
1  75.509800   72.90  
2  70.591048   68.66  
3  69.779874   50.24  
4  58.731894   40.17  


In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 90.00%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 60 of 60 rows (100.0%)


{'empirical_quantiles': [np.float64(0.06666666666666667),
  np.float64(0.18333333333333332),
  np.float64(0.2833333333333333),
  np.float64(0.3333333333333333),
  np.float64(0.3333333333333333),
  np.float64(0.43333333333333335),
  np.float64(0.5333333333333333),
  np.float64(0.65),
  np.float64(0.7),
  np.float64(0.8333333333333334),
  np.float64(0.8666666666666667),
  np.float64(0.9),
  np.float64(0.9666666666666667)],
 'average_interval_lengths': [np.float64(4.536349372255199),
  np.float64(7.717172555020045),
  np.float64(12.109593814549319)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.4404237949747833),
 'average_crps': np.float64(1.9255668028242712),
 'nll_values': array([1.93801961, 2.38452123, 4.65452088, 4.36985763, 2.98883621,
        2.66588935, 0.73549327, 2.08805745, 2.45820669, 4.17600825,
        1.88364797, 2.41405364, 1.66290664, 1.93205999, 2.6786342 ,
        2.13500628, 1.6189748 , 1.62128173, 2.82954915, 3.60761909,
        0.62598482, 1.3497

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 90.00%
The empirical coverage after calibration is: 83.33%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  70.826439  71.006193  71.850963  70.793915  70.163555  70.681976   
1  71.179363  73.441041  74.637544  75.283555  74.019878  76.664659   
2  65.282702  59.573695  60.542385  48.456878  49.754952  49.254673   
3  55.286514  57.201692  56.009869  58.303869  63.749434  63.543893   
4  32.815349  34.160893  32.479442  32.641981  35.206428  36.897902   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  71.925978  72.926140  71.959321  72.553274  72.344269  73.221631   
1  76.139848  76.370335  76.584808  77.358146  78.306413  78.701610   
2  51.291846  52.490465  55.990180  53.060648  56.138181  56.433994   
3  65.155729  66.995020  66.443649  67.199668  68.424681  66.917356   
4  39.680282  42.498509  43.733829  44.716435  43.192534  47.139294   

        0.95  Actual  
0  72.836318   70.07  
1  78.148143   72.90  
2  70.013735   68.66  
3  68.359284   50.24  
4  59.695312   40.17  


In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 91.67%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 60 of 60 rows (100.0%)


{'empirical_quantiles': [np.float64(0.06666666666666667),
  np.float64(0.21666666666666667),
  np.float64(0.2833333333333333),
  np.float64(0.45),
  np.float64(0.48333333333333334),
  np.float64(0.5666666666666667),
  np.float64(0.6833333333333333),
  np.float64(0.7666666666666667),
  np.float64(0.8),
  np.float64(0.85),
  np.float64(0.95),
  np.float64(0.9666666666666667),
  np.float64(0.9833333333333333)],
 'average_interval_lengths': [np.float64(5.142579090692109),
  np.float64(8.080006281463676),
  np.float64(13.149842853490265)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.9225204919409205),
 'average_crps': np.float64(2.10266630176558),
 'nll_values': array([ 2.83904628,  2.89768547,  4.66802859,  9.27801419,  3.0336922 ,
         2.7414082 ,  2.19718064,  2.35949556,  2.83921755,  2.76635213,
         1.78395375,  2.0829026 ,  1.90654088,  1.55593276,  2.74325406,
         0.47750961,  1.61022819,  1.76860571,  2.55114318,  2.68489564,
         1.90635378, 

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 86.67%
The empirical coverage after calibration is: 86.67%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 88 with best_epoch = 58 and best_val_0_pinball_0.05 = 0.66133



Early stopping occurred at epoch 73 with best_epoch = 43 and best_val_0_pinball_0.1 = 1.39251



Early stopping occurred at epoch 153 with best_epoch = 123 and best_val_0_pinball_0.15 = 0.85896



Early stopping occurred at epoch 166 with best_epoch = 136 and best_val_0_pinball_0.2 = 1.47902



Early stopping occurred at epoch 168 with best_epoch = 138 and best_val_0_pinball_0.3 = 1.58277



Early stopping occurred at epoch 158 with best_epoch = 128 and best_val_0_pinball_0.4 = 1.43235


... (3 outputs left out)


Early stopping occurred at epoch 84 with best_epoch = 54 and best_val_0_pinball_0.8 = 2.2202



Early stopping occurred at epoch 151 with best_epoch = 121 and best_val_0_pinball_0.85 = 0.88945



Early stopping occurred at epoch 85 with best_epoch = 55 and best_val_0_pinball_0.9 = 2.07435



Early stopping occurred at epoch 97 with best_epoch = 67 and best_val_0_pinball_0.95 = 0.73267
        0.05        0.1       0.15        0.2        0.3        0.4  \
0  61.106266  56.243271  62.595718  68.606018  67.458145  68.354088   
1  60.513100  59.058861  71.858864  74.358543  73.140671  73.794670   
2  37.242924  28.006348  51.761749  57.494812  63.240631  56.016014   
3  32.649464  31.181091  50.641815  57.558281  57.790787  51.375160   
4  29.836399  29.699852  33.791725  33.459003  33.129299  35.795734   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  70.492546  76.250168  70.489777  67.736526  70.150497  64.844391   
1  73.936890  75.659645  75.005150  76.439896  75.345352  71.125717   
2  64.768463  53.892384  65.499352  71.459648  68.993225  65.654884   
3  48.333626  59.542423  58.871758  55.712116  67.282303  69.403061   
4  37.999222  37.603935  38.758720  58.000114  46.052887  77.021690   

        0.95  Actual  
0  68.769211   70.07  
1  7

In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 100.00%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 60 of 60 rows (100.0%)


{'empirical_quantiles': [np.float64(0.0),
  np.float64(0.016666666666666666),
  np.float64(0.08333333333333333),
  np.float64(0.16666666666666666),
  np.float64(0.26666666666666666),
  np.float64(0.35),
  np.float64(0.36666666666666664),
  np.float64(0.45),
  np.float64(0.55),
  np.float64(0.6833333333333333),
  np.float64(0.8333333333333334),
  np.float64(0.9833333333333333),
  np.float64(1.0)],
 'average_interval_lengths': [np.float64(7.716572062174479),
  np.float64(22.072153727213543),
  np.float64(29.56647710800171)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(3.1296309963270916),
 'average_crps': np.float64(2.403405090316136),
 'nll_values': array([2.76401996, 2.59785664, 3.58399271, 3.56992067, 3.60102281,
        2.82987182, 2.97835072, 2.73739464, 2.88002298, 3.75652019,
        2.81891422, 2.90510913, 2.5541933 , 3.07933872, 3.39700037,
        2.38846812, 2.2498521 , 2.70481344, 3.79039983, 3.92009105,
        3.01642555, 2.56877885, 3.1911479 , 3.189671

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 100.00%
The empirical coverage after calibration is: 100.00%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU
Estimator 0/100, Train metric: 13.5823


Estimator 1/100, Train metric: 12.9859
Estimator 2/100, Train metric: 12.4356
Estimator 3/100, Train metric: 11.9029
Estimator 4/100, Train metric: 11.4019
Estimator 5/100, Train metric: 10.9165
Estimator 6/100, Train metric: 10.4506
Estimator 7/100, Train metric: 10.0087
Estimator 8/100, Train metric: 9.5896
Estimator 9/100, Train metric: 9.1922
Estimator 10/100, Train metric: 8.8209
Estimator 11/100, Train metric: 8.4728
Estimator 12/100, Train metric: 8.1313
Estimator 13/100, Train metric: 7.8142
Estimator 14/100, Train metric: 7.5028
Estimator 15/100, Train metric: 7.2011
Estimator 16/100, Train metric: 6.9205
Estimator 17/100, Train metric: 6.6474


Estimator 18/100, Train metric: 6.3886
Estimator 19/100, Train metric: 6.1544
Estimator 20/100, Train metric: 5.9238
Estimator 21/100, Train metric: 5.7004
Estimator 22/100, Train metric: 5.5008
Estimator 23/100, Train metric: 5.2968
Estimator 24/100, Train metric: 5.1073
Estimator 25/100, Train metric: 4.9373
Estimator 26/100, Train metric: 4.7665
Estimator 27/100, Train metric: 4.6124
Estimator 28/100, Train metric: 4.4596
Estimator 29/100, Train metric: 4.3087
Estimator 30/100, Train metric: 4.1734
Estimator 31/100, Train metric: 4.0334


Estimator 32/100, Train metric: 3.9122
Estimator 33/100, Train metric: 3.7774
Estimator 34/100, Train metric: 3.6682
Estimator 35/100, Train metric: 3.5549
Estimator 36/100, Train metric: 3.4547
Estimator 37/100, Train metric: 3.3491
Estimator 38/100, Train metric: 3.2461
Estimator 39/100, Train metric: 3.1490
Estimator 40/100, Train metric: 3.0611
Estimator 41/100, Train metric: 2.9699
Estimator 42/100, Train metric: 2.8949
Estimator 43/100, Train metric: 2.8201


Estimator 44/100, Train metric: 2.7370
Estimator 45/100, Train metric: 2.6646
Estimator 46/100, Train metric: 2.6019
Estimator 47/100, Train metric: 2.5310
Estimator 48/100, Train metric: 2.4609
Estimator 49/100, Train metric: 2.3927
Estimator 50/100, Train metric: 2.3280
Estimator 51/100, Train metric: 2.2702
Estimator 52/100, Train metric: 2.2238
Estimator 53/100, Train metric: 2.1783


Estimator 54/100, Train metric: 2.1239
Estimator 55/100, Train metric: 2.0783
Estimator 56/100, Train metric: 2.0355
Estimator 57/100, Train metric: 1.9918
Estimator 58/100, Train metric: 1.9570
Estimator 59/100, Train metric: 1.9246
Estimator 60/100, Train metric: 1.8874
Estimator 61/100, Train metric: 1.8525


... (2 outputs left out)

Estimator 78/100, Train metric: 1.4030
Estimator 79/100, Train metric: 1.3868
Estimator 80/100, Train metric: 1.3716
Estimator 81/100, Train metric: 1.3514
Estimator 82/100, Train metric: 1.3301
Estimator 83/100, Train metric: 1.3184
Estimator 84/100, Train metric: 1.3006


Estimator 85/100, Train metric: 1.2895
Estimator 86/100, Train metric: 1.2736
Estimator 87/100, Train metric: 1.2586
Estimator 88/100, Train metric: 1.2463
Estimator 89/100, Train metric: 1.2324
Estimator 90/100, Train metric: 1.2171
Estimator 91/100, Train metric: 1.1993
Estimator 92/100, Train metric: 1.1860


Estimator 93/100, Train metric: 1.1720
Estimator 94/100, Train metric: 1.1546
Estimator 95/100, Train metric: 1.1393
Estimator 96/100, Train metric: 1.1250
Estimator 97/100, Train metric: 1.1154
Estimator 98/100, Train metric: 1.0994
Estimator 99/100, Train metric: 1.0854


        0.05        0.1       0.15        0.2        0.3        0.4  \
0  71.724495  71.830032  71.876373  71.887863  71.916504  71.937004   
1  75.778854  75.795479  75.807220  75.814560  75.822655  75.827164   
2  61.542686  61.583126  61.628967  61.648964  61.679081  61.705555   
3  63.424385  63.504883  63.524632  63.541950  63.577969  63.606548   
4  39.024921  39.062656  39.088387  39.115185  39.136887  39.156132   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  71.948380  71.976768  72.003258  72.021545  72.035225  72.062599   
1  75.834106  75.839981  75.848602  75.855858  75.862564  75.871826   
2  61.728836  61.746223  61.758270  61.771015  61.792133  61.818886   
3  63.642715  63.668213  63.691410  63.728935  63.763088  63.769215   
4  39.186375  39.215805  39.237625  39.281803  39.312359  39.350285   

        0.95  Actual  
0  72.101967   70.07  
1  75.881409   72.90  
2  61.847839   68.66  
3  63.815464   50.24  
4  39.444481   40.17  


In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [70.07 72.9  68.66 50.24 40.17]
Coverage of 90% prediction interval: 6.67%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 60 rows (0.0%)


{'empirical_quantiles': [np.float64(0.55),
  np.float64(0.55),
  np.float64(0.55),
  np.float64(0.55),
  np.float64(0.5666666666666667),
  np.float64(0.5666666666666667),
  np.float64(0.5666666666666667),
  np.float64(0.5666666666666667),
  np.float64(0.5833333333333334),
  np.float64(0.5833333333333334),
  np.float64(0.5833333333333334),
  np.float64(0.6),
  np.float64(0.6166666666666667)],
 'average_interval_lengths': [np.float64(0.1048731803894043),
  np.float64(0.18524840672810872),
  np.float64(0.262213134765625)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(1572.3944392632459),
 'average_crps': np.float64(2.802985146109263),
 'nll_values': array([ 1.32746812e+02,  4.42668157e+03,  2.79019698e+03,  6.35416494e+03,
         2.86006777e+01,  1.29450890e+04, -2.45445565e+00, -1.15500512e+00,
         3.38100143e+03,  9.17670616e+01,  2.79257963e+03,  2.92269622e+02,
         3.88319274e+01,  2.21299184e+03, -6.47367930e-01,  7.35611206e+01,
         3.48762152e+02

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 13.33%
The empirical coverage after calibration is: 90.00%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/noise_uncertainty_analysis/quantile_regression/PGBM.xlsx")